<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [ ]:
# %pip install langchain langchain_community pypdf
# %pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [2]:
import os
import warnings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
warnings.filterwarnings('ignore')

<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [ ]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [16]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [20]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [21]:
load_dotenv()

True

In [22]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [23]:
from langchain_chroma import Chroma

In [24]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [29]:
user_question = "Why ASI and AGI are not using generally or verbally, but AI is most general?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [30]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
like intelligence, so an AI system with true 
human-like intelligence including perception, agency, consciousness, inten -
tions and maybe even emotions (see Turing 1950 or more popular Kurzweil 
2005). ANI, on the other hand, refers to very domain-specific AI systems being 
able to accomplish very specific tasks in very narrowly defined contexts only. 
Questions of agency or consciousness do not arise with ANI systems, they are 
merely tools, although potentially very powerful tools.
So far and tellingly, AGI can only be found in manifold media products 
within the fantasy or science fiction genre. Famous examples are Samantha in 
‘HER’ , Data in ‘Star Trek’ , HAL 9000 in ‘2001’ (based on a novel of the afore -
mentioned writer Arthur C. Clarke), Bishop in ‘ Aliens’ , the Terminator in the 
movie series of the same name or even the Maschinenmensch in ‘Metropolis’ 
(Hermann 2020).
Document 2:
n AI applications will be able to plan, solve and reason 
problems independently, 

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [31]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [ ]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [ ]:
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with technical questions about **Cytology**, specifically focusing on **Lung Cancer**.
Your answers must be based exclusively on provided content from technical books provided.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from the technical books:
'''
{formatted_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "The provided context does not contain this information."
   - Start explain cell morphology and then divide morphology in bulletpoints (nuclie, cytoplasm, background and other aspects to consider)
   - Follow by differential diagnosis
   - Lastly explain ancillary studies for malignant mesothelioma.

2. **Transparency**:
   - Reference the book's name and page numbers when providing information.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.

## TASK
1. Answer the user's question **directly** if possible.
2. Point the user to relevant parts of the documentation.
3. Provide the response in the following format:

## RESPONSE FORMAT
'''
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Source**:
• [Book Title], Page(s): [...]
'''
"""
print()

In [34]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [39]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.9,  # Increase creativity
    'max_tokens': 200,  # Allow for longer responses
    'top_p': 0.5,        # Use nucleus sampling
    'frequency_penalty': 2,  # Reduce repetition
    'presence_penalty': 2    # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [40]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [41]:
answer = completion.choices[0].message.content
print(answer)

'''
# Why ASI and AGI Are Not Commonly Used Terms Compared to AI

The terms Artificial Super Intelligence (ASI) and Artificial General Intelligence (AGI) are not commonly used in everyday language because they represent advanced stages of artificial intelligence that have not yet been realized. According to the provided context, AGI refers to a level of AI capable of performing any intellectual task that a human can do with flexibility across different domains. It is often depicted in science fiction but does not exist currently as practical technology.

Artificial Super Intelligence (ASI), on the other hand, represents an even more advanced form where machines surpass human intelligence across all fields including creativity, wisdom, and social skills—essentially making humans redundant for many tasks. This concept remains theoretical at this point.

In contrast, "AI" generally refers today mostly to what is known as Artificial Narrow Intelligence (ANI). ANI systems perform specific t

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [42]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [43]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [47]:
query_keywords = ["Vector", "RAG"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:1000]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
the creation of an artificial human like intelligence, so an AI system with true 
human-like intelligence including perception, agency, consciousness, inten -
tions and maybe even emotions (see Turing 1950 or more popular Kurzweil 
2005). ANI, on the other hand, refers to very domain-specific AI systems being 
able to accomplish very specific tasks in very narrowly defined contexts only. 
Questions of agency or consciousness do not arise with ANI systems, they are 
merely tools, although potentially very powerful tools.
So far and tellingly, AGI can only be found in manifold media products 
within the fantasy or science fiction genre. Famous examples are Samantha in 
‘HER’ , Data in ‘Star Trek’ , HAL 9000 in ‘2001’ (based on a novel of the afore -
mentioned writer Arthur C. Clarke), Bishop in ‘ Aliens’ , the Terminator in the 
movie series of the same name or even the Maschinenmensch in ‘Metropolis’ 
(Hermann 2020).
-----------------------------------------------------------

1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


In [55]:
import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
# from langchain.chat_models import ChatOpenAI
from termcolor import colored

# ============================
# 1. Your own text input (Atomic Habits excerpts)
# ============================
raw_text = """
Every action you take is a vote for the type of person you wish to become. No single instance will transform your beliefs, but as the votes build up, so does the evidence of your new identity.

Habits are the compound interest of self-improvement. The same way that money multiplies through compound interest, the effects of your habits multiply as you repeat them. They seem to make little difference on any given day and yet the impact they deliver over the months and years can be enormous.

Getting 1 percent better every day counts for a lot in the long-run. A slight change in your daily habits can guide your life to a very different destination.

The most effective way to change your habits is to focus not on what you want to achieve, but on who you wish to become. Your identity emerges out of your habits.

The ultimate form of intrinsic motivation is when a habit becomes part of your identity. It’s one thing to say I’m the type of person who wants this. It’s something very different to say I’m the type of person who is this.

The Four Laws of Behavior Change are a simple set of rules we can use to build better habits. They are:
1. Make it obvious (Cue)
2. Make it attractive (Craving)
3. Make it easy (Response)
4. Make it satisfying (Reward)

Inversion of the 1st Law: Make it invisible. Reduce exposure to the cues that trigger bad habits.
Inversion of the 2nd Law: Make it unattractive. Reframe your mindset to highlight the benefits of avoiding bad habits.
Inversion of the 3rd Law: Make it difficult. Increase friction for bad habits.
Inversion of the 4th Law: Make it unsatisfying. Create immediate pain or cost for breaking good habits.

Habit stacking: After [CURRENT HABIT], I will [NEW HABIT]. This links your new habit to an existing one.

Environment matters more than motivation. Design your environment to make the cues for good habits obvious and visible.

You do not rise to the level of your goals. You fall to the level of your systems.

Small changes often appear to make no difference until you cross a critical threshold. This is the Plateau of Latent Potential.

The aggregation of marginal gains: improving by 1% in many tiny ways leads to remarkable results over time.

Habits are dopamine-driven feedback loops. Every behavior that is rewarding releases dopamine, which motivates you to repeat it.

Never miss twice. Missing one day of a habit is an accident. Missing twice is the start of a new habit (a bad one).

The process of behavior change always starts with awareness. You need to know what is happening before you can change it.

The Goldilocks Rule: Humans experience peak motivation when working on tasks that are right on the edge of their current abilities. Not too hard. Not too easy. Just right.
"""

print(f"Loaded custom text: {len(raw_text)} characters")

# ============================
# 2. Split the text into chunks
# ============================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.create_documents([raw_text])

print(f"Split into {len(chunks)} chunks")

# ============================
# 3. Create or load vector database
# ============================
load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Run once to create DB, then comment the next 3 lines and uncomment the load line below
db = Chroma.from_documents(chunks, embeddings, persist_directory="./atomic_habits_chroma")
# db.persist()
print("Vector database created and saved")

# For future runs, use this instead:
# db = Chroma(persist_directory="./atomic_habits_chroma", embedding_function=embeddings)

# ============================
# 4. Retrieval & Prompt (same strict style you wanted)
# ============================
def ask_question(user_question: str):
    if not user_question.strip():
        print("Please ask a question!")
        return

    # Retrieve relevant chunks
    retrieved_docs = db.similarity_search(user_question, k=10)

    # Format context exactly like before
    context = "\n\n".join(
        [f"Content #{i+1}:\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)]
    )

    SYSTEM_PROMPT = """You are an expert on the book Atomic Habits by James Clear.
Answer strictly using only the provided context from the book text.
If the answer is not in the context, say: "This information is not present in the provided text from Atomic Habits."

Format your answer in clear Markdown.
"""

    full_prompt = f"""{SYSTEM_PROMPT}

USER QUESTION: {user_question}

CONTEXT FROM ATOMIC HABITS:
'''
{context}
'''
"""

    # ← THIS IS THE ONLY PART THAT CHANGES
    client = openai.OpenAI()   # uses your OPENAI_API_KEY from .env

    completion = client.chat.completions.create(
        model="gpt-4o",          # or "gpt-4o-mini" if you prefer
        temperature=0.3,
        max_tokens=800,
        messages=[
            {"role": "system", "content": "You are a helpful expert on Atomic Habits."},
            {"role": "user", "content": full_prompt}
        ]
    )

    answer = completion.choices[0].message.content

    print("\n" + "="*80)
    print("ANSWER (from Atomic Habits):")
    print("="*80)
    print(answer)

    # Show top snippet with highlighted keywords
    print("\nTop retrieved snippet:")
    snippet = retrieved_docs[0].page_content
    keywords = ["habit", "identity", "1%", "dopamine", "system", "environment"]
    for kw in keywords:
        snippet = snippet.replace(kw, colored(kw, 'green', attrs=['bold']))
    print(snippet)
    print("="*80)

# ============================
# 5. Automatically ask a sample question
# ============================
if __name__ == "__main__":
    sample_question = "According to Atomic Habits, what are the Four Laws of Behavior Change and their inversions for breaking bad habits?"
    print(f"\nAsking: {sample_question}\n")
    ask_question(sample_question)

Loaded custom text: 2756 characters
Split into 4 chunks
Vector database created and saved

Asking: According to Atomic Habits, what are the Four Laws of Behavior Change and their inversions for breaking bad habits?


ANSWER (from Atomic Habits):
According to *Atomic Habits* by James Clear, the Four Laws of Behavior Change are:

1. Make it obvious (Cue)
2. Make it attractive (Craving)
3. Make it easy (Response)
4. Make it satisfying (Reward)

The inversions of these laws for breaking bad habits are:

1. Make it invisible. Reduce exposure to the cues that trigger bad habits.
2. Make it unattractive. Reframe your mindset to highlight the benefits of avoiding bad habits.
3. Make it difficult. Increase friction for bad habits.
4. Make it unsatisfying. Create immediate pain or cost for breaking good habits.

Top retrieved snippet:
Inversion of the 1st Law: Make it invisible. Reduce exposure to the cues that trigger bad habits.
Inversion of the 2nd Law: Make it unattractive. Reframe your mind